# CrewAI + LangGraph Integration: Restaurant Social Media Marketing Pipeline

## Scenario

This builds on the pure-CrewAI restaurant marketing project. Instead of running the
5-agent CrewAI crew standalone, we wrap it inside a **LangGraph graph** as one node,
then add a second LangGraph node that post-processes the crew's output into two
separate, clean deliverables: the blog post and the social media content.

**Important:** there is no `main.py` and no `crewai run` anywhere in this notebook.
CrewAI is used purely as a Python library — `Crew(...).kickoff(...)` is just a
function call. LangGraph treats the entire CrewAI crew as a single "leaf" node: it
has no visibility into the five internal CrewAI agents, their tools, or their
context-chaining. From LangGraph's perspective, one node goes in, one result comes
back out.

### Graph shape

```
        +----------------------+       +--------------------------+
START ->| run_marketing_crew   | ----> | split_deliverables        | -> END
        | (CrewAI crew inside) |       | (parses blog vs. social)  |
        +----------------------+       +--------------------------+
```

- **`run_marketing_crew`** — calls `crew.kickoff(inputs=...)`, using the LangGraph
  state to parameterize the crew (e.g. which dish category to focus on), and stores
  the raw crew result back into state.
- **`split_deliverables`** — takes the raw crew result (a single blob of text
  containing both the blog post and the social posts) and splits it into two clean
  state fields: `blog_post` and `social_media_content`.

This mirrors a common real pattern: CrewAI is great at orchestrating a *sequential
content pipeline* internally, while LangGraph is used at a higher level to manage
*application state* and decide what happens with the result (save to a DB, send to
a review node, trigger a human-in-the-loop approval step, etc.).

## Solution Notebook

## Section 1 — Environment Setup

In [ ]:
import os
import sqlite3
from typing import TypedDict, Optional

from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from langgraph.graph import StateGraph, END

# Make sure your OPENAI_API_KEY (or other LLM provider key) is set in the environment
# before running this notebook, e.g.:
# os.environ["OPENAI_API_KEY"] = "sk-..."


## Section 2 — Mock Database and Tools (same as the pure-CrewAI project)

We reuse the exact same `restaurant_marketing.db` setup and tools from the
pure-CrewAI assignment: `dishes`, `nutrition`, and `produce` tables, each backed by
a CrewAI `@tool`.


In [ ]:
DB_PATH = "restaurant_marketing.db"


def setup_database():
    """Create and populate the mock restaurant marketing database."""
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE dishes (
            id INTEGER PRIMARY KEY,
            name TEXT,
            cuisine TEXT,
            main_ingredients TEXT,
            description TEXT
        )
    """)

    cursor.execute("""
        CREATE TABLE nutrition (
            dish_id INTEGER,
            calories INTEGER,
            protein_g REAL,
            carbs_g REAL,
            fat_g REAL,
            fiber_g REAL,
            notes TEXT,
            FOREIGN KEY (dish_id) REFERENCES dishes (id)
        )
    """)

    cursor.execute("""
        CREATE TABLE produce (
            ingredient TEXT,
            region TEXT,
            season TEXT,
            local_farm_source TEXT
        )
    """)

    dishes = [
        (1, "Butter Chicken Tacos", "Indian Fusion",
         "chicken, tomato, cream, corn tortilla, cilantro",
         "Classic North Indian butter chicken folded into soft corn tortillas, "
         "finished with a cilantro-mint crema."),
        (2, "Karachi Biryani Arancini", "Pakistani Fusion",
         "basmati rice, lamb, saffron, breadcrumbs, mozzarella",
         "Crispy fried biryani rice balls stuffed with slow-cooked lamb and a "
         "molten mozzarella center."),
        (3, "Lahori Chapli Burger", "Pakistani Fusion",
         "beef, pomegranate seeds, coriander, brioche bun",
         "A spiced Peshawari-style beef patty with pomegranate crunch, served on "
         "a toasted brioche bun."),
        (4, "Amritsari Kulcha Pizza", "Indian Fusion",
         "flatbread dough, spiced potato, paneer, mint chutney",
         "Amritsari kulcha reimagined as a stone-baked flatbread pizza topped with "
         "spiced potato and paneer."),
        (5, "Nihari Ramen", "Pakistani Fusion",
         "slow-braised beef shank, ramen noodles, chili oil, soft egg",
         "Overnight-braised Nihari broth ladled over ramen noodles, topped with "
         "chili oil and a soft-boiled egg."),
        (6, "Chana Chaat Bruschetta", "Indian Fusion",
         "chickpeas, tamarind, toasted baguette, yogurt, sev",
         "Tangy chickpea chaat piled onto toasted baguette slices, drizzled with "
         "tamarind and yogurt."),
    ]
    cursor.executemany("INSERT INTO dishes VALUES (?, ?, ?, ?, ?)", dishes)

    nutrition = [
        (1, 420, 28.0, 32.0, 19.0, 3.0, "Good source of protein; moderate calorie dish"),
        (2, 380, 16.0, 30.0, 21.0, 1.5, "Rich and indulgent; best as a shareable appetizer"),
        (3, 540, 30.0, 38.0, 27.0, 4.0, "High protein; pomegranate adds antioxidants"),
        (4, 460, 14.0, 52.0, 20.0, 5.0, "Vegetarian; good fiber from potato and flatbread"),
        (5, 610, 34.0, 48.0, 29.0, 2.5, "Hearty and calorie-dense; great cold-weather dish"),
        (6, 240, 9.0, 34.0, 7.0, 8.0, "High fiber, lighter option, vegetarian"),
    ]
    cursor.executemany("INSERT INTO nutrition VALUES (?, ?, ?, ?, ?, ?, ?)", nutrition)

    produce = [
        ("Tomato", "Local Valley Farms", "Summer", "Green Acres Co-op"),
        ("Cilantro", "Local Valley Farms", "Year-round (greenhouse)", "Green Acres Co-op"),
        ("Basmati Rice", "Imported - Punjab region", "Year-round", "Punjab Grain Importers"),
        ("Pomegranate", "Local Valley Farms", "Fall", "Red Hill Orchards"),
        ("Potato", "Local Valley Farms", "Fall/Winter", "Green Acres Co-op"),
        ("Paneer", "Local Dairy", "Year-round", "Meadowbrook Dairy"),
        ("Chickpeas", "Regional", "Year-round (dried)", "Heartland Pulses"),
        ("Mint", "Local Valley Farms", "Spring/Summer", "Green Acres Co-op"),
    ]
    cursor.executemany("INSERT INTO produce VALUES (?, ?, ?, ?)", produce)

    conn.commit()
    conn.close()
    print("Database setup complete:", DB_PATH)


setup_database()


In [ ]:
@tool("Trending Dishes Lookup")
def get_trending_dishes(cuisine_filter: str = "") -> str:
    """Returns the restaurant's new/trending fusion dishes. If cuisine_filter is
    provided (e.g. 'Indian Fusion' or 'Pakistani Fusion'), only dishes matching
    that cuisine are returned; otherwise all dishes are returned."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    if cuisine_filter:
        cursor.execute(
            "SELECT name, cuisine, main_ingredients, description FROM dishes "
            "WHERE LOWER(cuisine) LIKE LOWER(?)",
            (f"%{cuisine_filter}%",),
        )
    else:
        cursor.execute("SELECT name, cuisine, main_ingredients, description FROM dishes")
    rows = cursor.fetchall()
    conn.close()

    lines = []
    for name, cuisine, ingredients, desc in rows:
        lines.append(f"- {name} ({cuisine}): {desc} [Ingredients: {ingredients}]")
    return "\n".join(lines) if lines else "No matching dishes found."


@tool("Nutrition Info Lookup")
def get_nutrition_info(dish_name: str) -> str:
    """Given a dish name (or partial name), returns its nutrition facts:
    calories, protein, carbs, fat, and fiber."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute(
        """
        SELECT d.name, n.calories, n.protein_g, n.carbs_g, n.fat_g, n.fiber_g, n.notes
        FROM dishes d
        JOIN nutrition n ON d.id = n.dish_id
        WHERE LOWER(d.name) LIKE LOWER(?)
        """,
        (f"%{dish_name}%",),
    )
    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return f"No nutrition data found for '{dish_name}'."

    lines = []
    for name, cal, protein, carbs, fat, fiber, notes in rows:
        lines.append(
            f"{name}: {cal} kcal, {protein}g protein, {carbs}g carbs, "
            f"{fat}g fat, {fiber}g fiber. Note: {notes}"
        )
    return "\n".join(lines)


@tool("Local Produce Sourcing Lookup")
def get_local_produce_info() -> str:
    """Returns sourcing details for key ingredients: region, season, and
    local farm/supplier."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT ingredient, region, season, local_farm_source FROM produce")
    rows = cursor.fetchall()
    conn.close()

    lines = []
    for ingredient, region, season, source in rows:
        lines.append(f"- {ingredient}: sourced from {region} ({source}), in season: {season}")
    return "\n".join(lines)


## Section 3 — Agents (same 5-agent pipeline)

In [ ]:
trend_researcher = Agent(
    role="Culinary Trend Researcher",
    goal="Identify and describe the restaurant's newest fusion dishes for a "
         "marketing blog, focused on a specific cuisine when asked",
    backstory=(
        "You are a food journalist specializing in South Asian cuisine trends. "
        "You have a sharp eye for what makes a dish exciting to home cooks and "
        "diners alike, and you translate menu items into engaging descriptions."
    ),
    tools=[get_trending_dishes],
    verbose=True,
)

nutrition_analyst = Agent(
    role="Nutrition Analyst",
    goal="Provide accurate, easy-to-understand nutrition information for each "
         "featured dish",
    backstory=(
        "You are a registered dietitian who consults for restaurants. You "
        "translate nutrition facts into practical, non-judgmental language that "
        "helps diners make informed choices."
    ),
    tools=[get_nutrition_info],
    verbose=True,
)

sourcing_specialist = Agent(
    role="Local Sourcing Specialist",
    goal="Highlight where key ingredients come from, emphasizing local and "
         "seasonal sourcing",
    backstory=(
        "You are the restaurant's supply chain coordinator with deep "
        "relationships with regional farms and importers. You care about "
        "telling the story of where the food comes from."
    ),
    tools=[get_local_produce_info],
    verbose=True,
)

blog_writer = Agent(
    role="Blog Content Writer",
    goal="Write an engaging, well-structured restaurant blog post combining "
         "dish trends, nutrition, and sourcing",
    backstory=(
        "You are a restaurant marketing copywriter who has written dozens of "
        "high-performing blog posts. You know how to weave factual detail into "
        "a warm, appetizing narrative."
    ),
    verbose=True,
)

social_media_strategist = Agent(
    role="Social Media Strategist",
    goal="Repurpose the blog post into platform-native Facebook and Twitter/X "
         "content that drives restaurant visits",
    backstory=(
        "You are a social media manager for restaurant brands. You know "
        "Facebook rewards warm, story-driven posts with photos, while "
        "Twitter/X rewards short, punchy hooks and threads."
    ),
    verbose=True,
)


## Section 4 — Tasks

One addition versus the pure-CrewAI version: `research_task`'s description now
references `{focus_cuisine}`, a placeholder that gets filled in at runtime from
`crew.kickoff(inputs={...})`. This is how the LangGraph state flows *into* the crew.


In [ ]:
research_task = Task(
    description=(
        "Research the restaurant's new fusion dishes, focusing specifically on "
        "{focus_cuisine} dishes. For each dish, capture the name, cuisine "
        "influence, main ingredients, and what makes it noteworthy. Use the "
        "trending dishes tool, passing '{focus_cuisine}' as the cuisine_filter."
    ),
    expected_output="A list of the matching dishes with a 1-2 sentence hook for each.",
    agent=trend_researcher,
)

nutrition_task = Task(
    description=(
        "For each dish identified by the Culinary Trend Researcher, look up its "
        "nutrition facts (calories, protein, carbs, fat, fiber) and write a "
        "short, friendly nutrition summary."
    ),
    expected_output="A nutrition summary for each dish written in approachable language.",
    agent=nutrition_analyst,
    context=[research_task],
)

sourcing_task = Task(
    description=(
        "Using the dishes and their main ingredients, identify which "
        "ingredients are locally/seasonally sourced and describe the local "
        "farms or suppliers behind them."
    ),
    expected_output="A short write-up per dish (or per key ingredient) on local/seasonal sourcing.",
    agent=sourcing_specialist,
    context=[research_task],
)

blog_task = Task(
    description=(
        "Write a complete restaurant blog post (600-900 words) about the "
        "{focus_cuisine} dishes, titled something like 'New on the Menu'. "
        "Combine the dish descriptions, nutrition info, and local sourcing "
        "story into one cohesive, appetizing narrative with an intro, a "
        "section per dish, and a closing call-to-action to visit the "
        "restaurant. Format the blog post under a Markdown heading exactly "
        "'## Blog Post'."
    ),
    expected_output=(
        "A polished blog post starting with the heading '## Blog Post', "
        "followed by Markdown content with headers per dish."
    ),
    agent=blog_writer,
    context=[research_task, nutrition_task, sourcing_task],
)

social_media_task = Task(
    description=(
        "Based on the finished blog post, create: "
        "(1) one Facebook post (150-200 words, warm tone, includes a call to "
        "action and 3-5 hashtags) under the heading '## Facebook Post', and "
        "(2) a Twitter/X thread of 4-5 tweets (each under 280 characters) "
        "under the heading '## Twitter Thread', teasing the new dishes, with a "
        "final tweet linking back to the blog and a call to action. Keep the "
        "two headings exactly as given so they can be parsed programmatically."
    ),
    expected_output=(
        "Two sections, each starting with its exact Markdown heading: "
        "'## Facebook Post' and '## Twitter Thread'."
    ),
    agent=social_media_strategist,
    context=[blog_task],
)


## Section 5 — Assemble the Crew (no `main.py`, no `crewai run`)

In [ ]:
crew = Crew(
    agents=[
        trend_researcher,
        nutrition_analyst,
        sourcing_specialist,
        blog_writer,
        social_media_strategist,
    ],
    tasks=[
        research_task,
        nutrition_task,
        sourcing_task,
        blog_task,
        social_media_task,
    ],
    process=Process.sequential,
    verbose=True,
)

# Note: we do NOT call crew.kickoff() here. Kicking off the crew happens inside
# the LangGraph node in Section 7, so that the crew's inputs come from LangGraph
# state rather than being hardcoded here.
print("Crew assembled:", [agent.role for agent in crew.agents])


## Section 6 — Define the LangGraph State

The state is the single shared dict that flows between LangGraph nodes. It needs to
carry whatever comes in from the caller (`focus_cuisine`), the raw output from the
CrewAI crew (`raw_crew_output`), and the **three** split-out deliverables produced by
the second node: `blog_post`, `facebook_post`, and `twitter_thread`.

Note this is three separate fields, not two. An earlier version of this node lumped
Facebook and Twitter into a single `social_media_content` field by taking
"everything from the Facebook marker to the end of the string" — which meant that if
the Twitter section came out mangled, missing, or the marker didn't match exactly,
it silently vanished into (or was missing from) that combined blob with no way to
tell which one failed. Splitting into three independently-located markers fixes
that: each section is found on its own, so a missing/malformed Twitter section
shows up as an empty `twitter_thread` field you can actually detect and handle,
rather than disappearing inside a bigger string.


In [ ]:
class MarketingState(TypedDict):
    focus_cuisine: str            # input: e.g. "Indian Fusion" or "Pakistani Fusion"
    raw_crew_output: Optional[str]  # output of crew.kickoff(), set by node 1
    blog_post: Optional[str]        # parsed out by node 2
    facebook_post: Optional[str]    # parsed out by node 2
    twitter_thread: Optional[str]   # parsed out by node 2


## Section 7 — Node 1: `run_marketing_crew`

This node's entire job is: take relevant fields out of LangGraph state, pass them
into `crew.kickoff(inputs=...)`, and write the crew's result back into state. It
does NOT know or care that internally this triggers 5 sequential CrewAI agents.


In [ ]:
def run_marketing_crew(state: MarketingState) -> dict:
    """LangGraph node: runs the entire CrewAI crew and stores its raw output.

    This is the ONLY place crew.kickoff() gets called. Everything about the 5
    internal CrewAI agents, their tools, and their sequential context-passing
    is invisible to LangGraph -- it just sees this one node run and return.
    """
    result = crew.kickoff(inputs={"focus_cuisine": state["focus_cuisine"]})
    return {"raw_crew_output": str(result)}


## Section 8 — Node 2: `split_deliverables`

This node has no CrewAI involvement at all — it's a plain LangGraph node that
post-processes `raw_crew_output` into three clean fields: `blog_post`,
`facebook_post`, and `twitter_thread`. This is the kind of task LangGraph is good at
that CrewAI's `Process.sequential` doesn't give you directly: free-form control flow
over the crew's result.

Each marker (`## Blog Post`, `## Facebook Post`, `## Twitter Thread`) is located
**independently** with `str.find()`, then the markers actually found are sorted by
position and each section runs from its own marker to the start of the next one
found (or to the end of the string for the last one). This way a missing or
misspelled marker only empties out that one field instead of corrupting or merging
adjacent sections.


In [ ]:
def split_deliverables(state: MarketingState) -> dict:
    """LangGraph node: splits the crew's raw output into blog_post,
    facebook_post, and twitter_thread using the '## Blog Post' /
    '## Facebook Post' / '## Twitter Thread' markers the tasks were
    instructed to produce.

    Each marker is located independently, then the markers that were
    actually found are sorted by their position in the text. Each section
    runs from its own marker up to the next FOUND marker (not just the next
    marker in a fixed list) -- so if, say, the Twitter marker never shows up,
    the Facebook section still correctly extends to the end of the string
    instead of accidentally swallowing nothing or throwing an index error,
    and twitter_thread comes back as an empty string you can check for and
    handle (e.g. retry, log a warning) instead of silently losing data.
    """
    raw = state.get("raw_crew_output") or ""

    markers = {
        "blog_post": "## Blog Post",
        "facebook_post": "## Facebook Post",
        "twitter_thread": "## Twitter Thread",
    }

    positions = {name: raw.find(marker) for name, marker in markers.items()}
    found = sorted((pos, name) for name, pos in positions.items() if pos != -1)

    result = {"blog_post": "", "facebook_post": "", "twitter_thread": ""}

    if not found:
        # Fallback: no markers matched at all (e.g. LLM ignored formatting
        # instructions entirely) -- keep everything as blog_post so nothing
        # is silently discarded.
        result["blog_post"] = raw
        return result

    for i, (pos, name) in enumerate(found):
        end = found[i + 1][0] if i + 1 < len(found) else len(raw)
        result[name] = raw[pos:end].strip()

    return result


## Section 9 — Build and Compile the Graph

In [ ]:
graph = StateGraph(MarketingState)

graph.add_node("run_marketing_crew", run_marketing_crew)
graph.add_node("split_deliverables", split_deliverables)

graph.set_entry_point("run_marketing_crew")
graph.add_edge("run_marketing_crew", "split_deliverables")
graph.add_edge("split_deliverables", END)

app = graph.compile()
print("Graph compiled with nodes:", list(app.get_graph().nodes))


## Section 10 — Invoke the Graph

In [ ]:
final_state = app.invoke({"focus_cuisine": "Pakistani Fusion"})

print("=== BLOG POST ===\n")
print(final_state["blog_post"])
print("\n=== FACEBOOK POST ===\n")
print(final_state["facebook_post"])
print("\n=== TWITTER THREAD ===\n")
print(final_state["twitter_thread"])

if not final_state["twitter_thread"]:
    print("\n[WARNING] twitter_thread came back empty -- the model likely "
          "didn't emit the '## Twitter Thread' heading. Check raw_crew_output.")
